# License Plate Segmentor — YOLOv8n-seg Training
Trains a **YOLOv8n-seg** model on the Open Images V7 'Vehicle registration plate' class.
Instance segmentation masks are used when available; bounding-box polygons are the fallback.
No account needed. Uses Colab T4 GPU (~20 min training).

**Steps:**
1. Install dependencies
2. Download dataset from Open Images V7 (with segmentation masks)
3. Convert to YOLO segmentation format
4. Train YOLOv8n-seg
5. Export to ONNX
6. Save to Google Drive

**Place the output at:** `backend/models/plate-segmentor.onnx`

## 0. Enable GPU
Before running: **Runtime → Change runtime type → T4 GPU**

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. Install dependencies

In [ ]:
!pip install -q ultralytics fiftyone opencv-python-headless

## 2. Download license plate data from Open Images V7
Request **both** detections and segmentations.  
Fiftyone populates `det.mask` (a boolean crop-sized array) when a segmentation mask exists for that instance.

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz

train_dataset = foz.load_zoo_dataset(
    'open-images-v7',
    split='train',
    label_types=['detections', 'segmentations'],
    classes=['Vehicle registration plate'],
    max_samples=3000,
    dataset_name='plates_train',
)

val_dataset = foz.load_zoo_dataset(
    'open-images-v7',
    split='validation',
    label_types=['detections', 'segmentations'],
    classes=['Vehicle registration plate'],
    max_samples=500,
    dataset_name='plates_val',
)

print(f'Train samples: {len(train_dataset)}')
print(f'Val samples:   {len(val_dataset)}')

## 3. Convert to YOLO segmentation format
Each label file line: `0 x1 y1 x2 y2 ...` (class 0, normalised polygon coords).

- If `det.mask` exists → extract contour polygon from the instance mask  
- Otherwise → use the 4 bbox corners as a rectangular polygon (still valid for segmentation training)

In [ ]:
import cv2
import os
import shutil
import numpy as np
from PIL import Image

DATASET_DIR = '/content/plate_seg_dataset'

# Auto-detect the label field (Open Images uses 'ground_truth', not 'detections')
sample = train_dataset.first()
label_field = None
for field_name, field in sample.iter_fields():
    if hasattr(field, 'detections'):
        label_field = field_name
        break

if label_field is None:
    print('Available fields:', list(sample.field_names))
    raise ValueError('Could not find a detections field.')

print(f'Using label field: {label_field!r}')

# Check if any masks are present
sample_dets = sample[label_field]
has_masks = (
    sample_dets is not None
    and len(sample_dets.detections) > 0
    and getattr(sample_dets.detections[0], 'mask', None) is not None
)
print(f'Instance masks available: {has_masks}')


def det_to_polygon(det):
    """
    Convert a fiftyone Detection to a flat list of normalised polygon coords.
    Uses the instance mask contour when available, bbox corners otherwise.
    """
    bx, by, bw, bh = det.bounding_box  # top-left x, y, width, height (normalised)

    if has_masks and getattr(det, 'mask', None) is not None:
        mask = (det.mask.astype(np.uint8)) * 255
        cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if cnts:
            cnt = max(cnts, key=cv2.contourArea)
            eps = 0.02 * cv2.arcLength(cnt, True)
            approx = cv2.approxPolyDP(cnt, eps, True).reshape(-1, 2)
            mh, mw = mask.shape
            pts = []
            for px, py in approx:
                pts += [bx + (px / mw) * bw, by + (py / mh) * bh]
            if len(pts) >= 6:
                return pts

    # Fallback: 4 bbox corners
    return [bx, by, bx + bw, by, bx + bw, by + bh, bx, by + bh]


def process_split(dataset, split):
    img_dir = os.path.join(DATASET_DIR, split, 'images')
    lbl_dir = os.path.join(DATASET_DIR, split, 'labels')
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)

    for sample in dataset:
        # Copy image
        dst = os.path.join(img_dir, os.path.basename(sample.filepath))
        shutil.copy(sample.filepath, dst)

        dets = sample[label_field]
        if dets is None:
            continue

        lines = []
        for det in dets.detections:
            pts = det_to_polygon(det)
            pts = [max(0.0, min(1.0, p)) for p in pts]
            lines.append('0 ' + ' '.join(f'{p:.6f}' for p in pts))

        if lines:
            stem = os.path.splitext(os.path.basename(sample.filepath))[0]
            with open(os.path.join(lbl_dir, stem + '.txt'), 'w') as f:
                f.write('\n'.join(lines) + '\n')

    print(f'{split}: {len(dataset)} images')


process_split(train_dataset, 'train')
process_split(val_dataset,   'val')
print('Done.')

In [ ]:
# Write dataset.yaml
yaml_content = f"""path: {DATASET_DIR}
train: train/images
val: val/images

nc: 1
names: ['plate']
"""

yaml_path = os.path.join(DATASET_DIR, 'dataset.yaml')
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print('dataset.yaml written')
print(yaml_content)

## 4. Train YOLOv8n-seg

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n-seg.pt')  # nano segmentation — pretrained on COCO

results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project='/content/runs',
    name='plate_segmentor',
    patience=10,
    exist_ok=True,
)

print('Training done. Best weights:', results.save_dir)

## 5. Evaluate on validation set

In [ ]:
best_weights = '/content/runs/plate_segmentor/weights/best.pt'
trained = YOLO(best_weights)
metrics = trained.val(data=yaml_path)
print(f'Box  mAP50:    {metrics.box.map50:.3f}')
print(f'Box  mAP50-95: {metrics.box.map:.3f}')
print(f'Mask mAP50:    {metrics.seg.map50:.3f}')
print(f'Mask mAP50-95: {metrics.seg.map:.3f}')

## 6. Export to ONNX

In [ ]:
trained.export(
    format='onnx',
    imgsz=640,
    simplify=True,
    opset=12,
)

onnx_path = best_weights.replace('.pt', '.onnx')
print('ONNX exported to:', onnx_path)
!ls -lh {onnx_path}

## 7. Save to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
dest = '/content/drive/MyDrive/plate-segmentor.onnx'
shutil.copy(onnx_path, dest)
print('Saved to Google Drive:', dest)
print('Place the file at: backend/models/plate-segmentor.onnx')

## Done
Download `plate-segmentor.onnx` from Google Drive and place it at:
```
backend/models/plate-segmentor.onnx
```
The backend will automatically use it for detect → warp → OCR.